# Dollar-Neutral Pair Trade: TXN / KVUE

A classic **dollar-neutral** pair trade: long TXN (Texas Instruments), short KVUE (Kenvue),
with BIL as collateral.

The strategy uses a **parent/child portfolio tree** with controlled book sizes via a hedge ratio:
- Long leg: TXN (equal weight)
- Short leg: KVUE (equal weight, negative)
- Parent controls the 1:1.135 hedge ratio between legs

This notebook demonstrates:
- Parent/child portfolio tree for long/short strategies
- `Weigh.Equally(short=True)` for short positions
- `Weigh.Ratio` for controlling book sizes
- Dollar-neutral construction

**Requires network**: KVUE (listed Sept 2023) and TXN are fetched live via YFinance.

In [ ]:
import tiportfolio as ti

# Hedge ratio derived from OLS regression of TXN returns on KVUE returns
RATIO = 1.135
LONG_NAME = "long_txn"
SHORT_NAME = "short_kvue"

START = "2023-09-01"
END = "2024-12-31"

## 1. Load Data

KVUE was listed in September 2023, so the date range starts there.
Data is fetched live from YFinance.

In [ ]:
data = ti.fetch_data(["TXN", "KVUE", "BIL"], start=START, end=END)

for ticker, df in data.items():
    print(f"{ticker}: {df.shape[0]} rows, {df.index[0].date()} \u2192 {df.index[-1].date()}")

## 2. Dollar-Neutral Strategy

The parent/child tree architecture:

```
dollar_neutral (parent)
├── long_txn (child)  — holds TXN with positive weight
└── short_kvue (child) — holds KVUE with negative weight
```

The parent allocates capital to each child based on the hedge ratio.
`Weigh.Equally(short=True)` makes weights negative for the short leg.

In [ ]:
long = ti.Portfolio(
    LONG_NAME,
    [
        ti.Select.All(),
        ti.Weigh.Equally(),
        ti.Action.Rebalance(),
    ],
    ["TXN"],
)

short = ti.Portfolio(
    SHORT_NAME,
    [
        ti.Select.All(),
        ti.Weigh.Equally(short=True),
        ti.Action.Rebalance(),
    ],
    ["KVUE"],
)

# Parent allocates between long and short legs
# With RATIO=1.135: long gets ~46.8%, short gets ~53.2%
long_book = 1.0 / (1.0 + RATIO)
short_book = RATIO / (1.0 + RATIO)

dollar_neutral = ti.Portfolio(
    "dollar_neutral",
    [
        ti.Signal.Monthly(),
        ti.Select.All(),
        ti.Weigh.Ratio(weights={LONG_NAME: long_book, SHORT_NAME: short_book}),
        ti.Action.Rebalance(),
    ],
    [long, short],
)

print(f"Long book size:  {long_book:.3f}")
print(f"Short book size: {short_book:.3f}")

In [ ]:
result = ti.run(ti.Backtest(dollar_neutral, data))

## 3. Results

In [ ]:
result.summary()

In [ ]:
result[0].plot_interactive()

In [ ]:
result[0].plot_security_weights()

In [ ]:
print(f"Total trades: {len(result[0].trades)}")
result[0].trades.sample(3)

## 4. Baseline Comparisons

Compare the dollar-neutral pair against:
1. **Long TXN Only** \u2014 100% TXN buy-and-hold
2. **50/50 TXN + BIL** \u2014 half in TXN, half in BIL

In [ ]:
txn_only = ti.Portfolio(
    "txn_only",
    [ti.Signal.Once(), ti.Select.All(), ti.Weigh.Equally(), ti.Action.Rebalance()],
    ["TXN"],
)

txn_bil_5050 = ti.Portfolio(
    "txn_bil_5050",
    [
        ti.Signal.Monthly(),
        ti.Select.All(),
        ti.Weigh.Ratio(weights={"TXN": 0.5, "BIL": 0.5}),
        ti.Action.Rebalance(),
    ],
    ["TXN", "BIL"],
)

comparison = ti.run(
    ti.Backtest(dollar_neutral, data),
    ti.Backtest(txn_only, data),
    ti.Backtest(txn_bil_5050, data),
)

In [ ]:
comparison.summary()

In [ ]:
comparison.plot_interactive()